<h1 style="text-align: center;">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro do Desenvolvendo Juntos</h1>
<br>
<br>

**Componente:** Fundamentos de Ambientes e Arquitetura de Dados
<br>
**Unidade Curricular:** Introdução ao PySpark e Processamento Distribuído
<br>
**Tema da Semana:** Transformações em DataFrames
<br>
**Semana 18 – Aula 2**
<br>
**Aula 2:** Tipos e nulos


## Objetivo
Compreender como ajustar tipos de dados e tratar valores ausentes em colunas, utilizando operações básicas de preparação de dados.

Durante a atividade vamos utilizar as seguintes funções:

- `cast()`
- `dropna()`
- `fillna()`
- `replace()`

Essas operações são importantes porque ajudam a deixar os dados mais organizados e adequados para análises posteriores.

## Preparando o ambiente

Antes de trabalhar com dados no PySpark, precisamos iniciar uma sessão Spark e importar as funções que serão utilizadas.

A `SparkSession` é o ponto de entrada para criar e manipular DataFrames no PySpark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tipos_e_nulos_Aula2") \
    .getOrCreate()


## Leitura do arquivo de dados

Agora vamos carregar um arquivo CSV com informações de produtos.

Esse conjunto de dados foi organizado para permitir a prática de operações comuns de preparação de dados, como conversão de tipos, identificação de valores ausentes e padronização de informações.

Arquivo utilizado:

`DADOSANO2C6B3S18A2_auxiliar.csv`


In [2]:
df = spark.read.csv(
    "DADOSANO2C6B3S18A2_auxiliar.csv",
    header=True,
    inferSchema=True
)

df.show()


+------------+------------+-----+-------+
|     produto|   categoria|preco|estoque|
+------------+------------+-----+-------+
|     Teclado| Periféricos|  120|     35|
|       Mouse| Periféricos|   60|   NULL|
|     Monitor|    Hardware|  900|     12|
|     Headset|desconhecido|  150|     28|
|    Notebook|    Hardware| 3500|   NULL|
|Caixa de som|       Áudio|  220|     15|
+------------+------------+-----+-------+



## Observando a estrutura do DataFrame

Antes de transformar os dados, vamos observar como o PySpark identificou os tipos de cada coluna.

O método `printSchema()` mostra:

- os nomes das colunas;
- os tipos de dados;
- a estrutura geral do DataFrame.


In [4]:
df.printSchema()



root
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: integer (nullable = true)
 |-- estoque: integer (nullable = true)



## Convertendo tipos com `cast()`

Em alguns casos, uma coluna que representa números pode ser lida como texto.

Quando isso acontece, precisamos converter o tipo da coluna para permitir cálculos e análises corretamente.

Neste exemplo, vamos converter a coluna `preco` para o tipo `double`.


In [5]:
df = df.withColumn(
    "preco",
    col("preco").cast("double")
)

df.printSchema()
df.show()



root
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: double (nullable = true)
 |-- estoque: integer (nullable = true)

+------------+------------+------+-------+
|     produto|   categoria| preco|estoque|
+------------+------------+------+-------+
|     Teclado| Periféricos| 120.0|     35|
|       Mouse| Periféricos|  60.0|   NULL|
|     Monitor|    Hardware| 900.0|     12|
|     Headset|desconhecido| 150.0|     28|
|    Notebook|    Hardware|3500.0|   NULL|
|Caixa de som|       Áudio| 220.0|     15|
+------------+------------+------+-------+



## Identificando valores nulos

Agora vamos observar o que acontece com registros que possuem valores ausentes.

No PySpark, esses valores aparecem como `null`.

Eles podem prejudicar cálculos, filtros e outras operações se não forem tratados corretamente.



In [6]:
df.show()

+------------+------------+------+-------+
|     produto|   categoria| preco|estoque|
+------------+------------+------+-------+
|     Teclado| Periféricos| 120.0|     35|
|       Mouse| Periféricos|  60.0|   NULL|
|     Monitor|    Hardware| 900.0|     12|
|     Headset|desconhecido| 150.0|     28|
|    Notebook|    Hardware|3500.0|   NULL|
|Caixa de som|       Áudio| 220.0|     15|
+------------+------------+------+-------+



## Removendo registros com valores nulos usando `dropna()`

Uma forma de tratar valores ausentes é remover as linhas que possuem campos nulos.

Isso pode ser útil em alguns casos, mas deve ser feito com cuidado, pois reduz a quantidade de registros no DataFrame.



In [7]:
df_sem_nulos = df.dropna()

df_sem_nulos.show()


+------------+------------+-----+-------+
|     produto|   categoria|preco|estoque|
+------------+------------+-----+-------+
|     Teclado| Periféricos|120.0|     35|
|     Monitor|    Hardware|900.0|     12|
|     Headset|desconhecido|150.0|     28|
|Caixa de som|       Áudio|220.0|     15|
+------------+------------+-----+-------+



## Substituindo valores ausentes com `fillna()`

Em vez de remover registros, também podemos substituir os valores ausentes.

Neste exemplo, vamos preencher valores nulos da coluna `estoque` com `0`.

Isso permite manter o registro no conjunto de dados.


In [8]:
df_preenchido = df.fillna({
    "estoque": 0
})

df_preenchido.show()


+------------+------------+------+-------+
|     produto|   categoria| preco|estoque|
+------------+------------+------+-------+
|     Teclado| Periféricos| 120.0|     35|
|       Mouse| Periféricos|  60.0|      0|
|     Monitor|    Hardware| 900.0|     12|
|     Headset|desconhecido| 150.0|     28|
|    Notebook|    Hardware|3500.0|      0|
|Caixa de som|       Áudio| 220.0|     15|
+------------+------------+------+-------+



## Substituindo valores específicos com `replace()`

Além dos valores nulos, às vezes precisamos padronizar informações do conjunto de dados.

A função `replace()` permite trocar um valor específico por outro.

Neste exemplo, vamos substituir o valor `"desconhecido"` por `"não informado"` na coluna `categoria`.


In [9]:
df_ajustado = df_preenchido.replace(
    "desconhecido",
    "não informado",
    subset=["categoria"]
)

df_ajustado.show()


+------------+-------------+------+-------+
|     produto|    categoria| preco|estoque|
+------------+-------------+------+-------+
|     Teclado|  Periféricos| 120.0|     35|
|       Mouse|  Periféricos|  60.0|      0|
|     Monitor|     Hardware| 900.0|     12|
|     Headset|não informado| 150.0|     28|
|    Notebook|     Hardware|3500.0|      0|
|Caixa de som|        Áudio| 220.0|     15|
+------------+-------------+------+-------+



## Síntese da atividade

Ao longo desta atividade, observamos que a preparação de dados envolve diferentes decisões.

Nesta prática, utilizamos:

- `cast()` para ajustar o tipo de uma coluna;
- `dropna()` para remover registros com valores ausentes;
- `fillna()` para substituir valores nulos;
- `replace()` para padronizar valores específicos.

Essas operações ajudam a deixar o DataFrame mais adequado para análises e transformações posteriores.
